# 05 Walk-forward validation

**Author:** Rowan Walker

This notebook presents an implementation of a walk-forward validation. Model training followed by subsequent out-of-sample back testing aims to demonstrate the robustness of the model over on a walk-forward basis; a critical component in determining the robustness of future returns.

### 5.1 Imports

In [5]:
import numpy as np
import pandas as pd

from src.utils.seed import set_global_seed
from src.training.train import generate_walk_forward_dates, walk_forward_validation

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch")
warnings.filterwarnings("ignore", category=FutureWarning, module="torch")
warnings.filterwarnings("ignore", category=UserWarning, module="hmmlearn")

%reload_ext autoreload
%autoreload 2

# Global seed setting for reproducibility
set_global_seed()

### 5.2 Walk forward validation

**Purpose**: This walk-forward validation has been performed to demonstrate the models robustness in a continuous test and train window when walking forward from historic data up to the most recent. This mimics the real-life implementation whereby the model would be re-trained regularly, therefore ensuring that the strategy performs well on multiple out-of-sample windows under different economic conditions, regimes and timepoints is one of the most powerful tests of model robustness. Walk-forward dates are generated on a rolling basis where trainining window length is equal to 5 years and the out-of-sample testing period is 1 year. A common back test parameter grid has been used to minimise the impact of potential data-snooping bias where excess parameter tuning may cause overfitting even on an out-of-sample window.

#### Setting a common parameter grid for the back test

In [6]:
param_grid = {
    'long_threshold': 0.6, 
    'short_threshold': 0.4, 
    'target_vol': 0.2,  
    'slippage': 1.0,
    'commission': 1.0,
    'take_profit': 0.16,
    'stop_loss': -0.02,
    'max_hold_days': 18.0,
    'max_drawdown': 0.2,
    'leverage': 2.0}

#### Results

In [7]:
walk_forward_validation(
    param_grid=param_grid,
    hold_days=(1, 5, 21),
    start='2010-01-01',
    end='2025-12-31',
    train_length=5,
    test_length=1
)

Beginning walk-forward validation 1 of 10. Train Window: 2010-01-01 to 2015-01-01.
Walk-forward validation 1 finished. Time taken = 96.53 seconds.

Beginning walk-forward validation 2 of 10. Train Window: 2011-01-01 to 2016-01-01.
Walk-forward validation 2 finished. Time taken = 95.89 seconds.

Beginning walk-forward validation 3 of 10. Train Window: 2012-01-01 to 2017-01-01.
Walk-forward validation 3 finished. Time taken = 99.61 seconds.

Beginning walk-forward validation 4 of 10. Train Window: 2013-01-01 to 2018-01-01.
Walk-forward validation 4 finished. Time taken = 100.76 seconds.

Beginning walk-forward validation 5 of 10. Train Window: 2014-01-01 to 2019-01-01.
Walk-forward validation 5 finished. Time taken = 101.81 seconds.

Beginning walk-forward validation 6 of 10. Train Window: 2015-01-01 to 2020-01-01.
Walk-forward validation 6 finished. Time taken = 104.62 seconds.

Beginning walk-forward validation 7 of 10. Train Window: 2016-01-01 to 2021-01-01.
Walk-forward validation 7 

,Sharpe
Train Start,
2010-01-01,3.783685
2011-01-01,3.419369
2012-01-01,2.545877
2013-01-01,1.724125
2014-01-01,3.148255
2015-01-01,3.058062
2016-01-01,3.144919
2017-01-01,2.616048
2018-01-01,2.066018


**Conclusion**: The model’s performance is moderately sensitive to the chosen training start date. Across the 2010–2019 range, Sharpe ratios vary between ~1.7 and ~3.8, with a mean around 3.05.

The model performs strongest when trained on longer histories (2010–2011), suggesting it benefits from diverse market regimes. There is a noticeable performance trough around 2012–2013, indicating sensitivity to the specific data regime used for training. From 2014 onward, Sharpe stabilises around 3.0–3.15, showing the model is reasonably robust in more recent periods.

The strategy appears structurally sound, but its performance depends on capturing sufficient regime diversity in the training window. This reinforces the importance of including multiple market environments during training and treating regime features primarily as regularising signals rather than dominant predictors.